In [ ]:
import pandas as pd
import re
import unicodedata
from rapidfuzz import fuzz

ref = pd.read_csv("dim_location_20260316_v2 2.csv")

# reference dataset (barangay, city, province, region)
df = pd.read_csv("ps_address_mapping_0413 - Real Edit(in).csv")

# alias table: pattern | replacement
aliases = pd.read_csv("aliases.csv", encoding='latin1')


def normalize(text):
    if pd.isna(text):
        return ""
    
    # lowercase
    text = text.lower()
    
    # remove accents (ñ → n)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    
    # remove special characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
# normalize alias patterns
aliases['pattern_clean'] = aliases['pattern'].apply(normalize)
aliases['replacement_clean'] = aliases['replacement'].apply(normalize)

# convert to dictionary
alias_dict = dict(zip(aliases['pattern_clean'], aliases['replacement_clean']))

# compile regex patterns (for speed)
alias_patterns = {
    re.compile(rf'\b{re.escape(k)}\b'): v
    for k, v in alias_dict.items()
}

In [ ]:
def apply_aliases(text):
    for pattern, replacement in alias_patterns.items():
        text = pattern.sub(replacement, text)
    return text

In [ ]:
def normalize_full(text):
    text = normalize(text)
    text = apply_aliases(text)
    return text

In [ ]:
df['addr_clean'] = df['original_address'].apply(normalize)
df['city_clean'] = df['city_name'].apply(normalize)
df['prov_clean'] = df['province_name'].apply(normalize)

ref['city_clean'] = ref['city_name'].apply(normalize_full)
ref['prov_clean'] = ref['province_name'].apply(normalize_full)

In [ ]:
from rapidfuzz import fuzz

def best_city_match(addr, ref, threshold=80):
    scores = ref['city_clean'].apply(
        lambda x: fuzz.partial_ratio(x, addr)
    )
    
    best_idx = scores.idxmax()
    best_score = scores[best_idx]
    
    return pd.Series({
        'city_match_fuzzy': best_score >= threshold,
        'matched_city': ref.loc[best_idx, 'city_name'],
        'city_score': best_score
    })

In [ ]:
def best_province_match(addr, ref, threshold=80):
    scores = ref['prov_clean'].apply(
        lambda x: fuzz.partial_ratio(x, addr)
    )
    
    best_idx = scores.idxmax()
    best_score = scores[best_idx]
    
    return pd.Series({
        'province_match_fuzzy': best_score >= threshold,
        'matched_province': ref.loc[best_idx, 'province_name'],
        'province_score': best_score
    })

In [ ]:
print(ref.columns)

In [ ]:
df[['city_match_fuzzy', 'matched_city', 'city_score']] = df.apply(
    lambda x: best_city_match(x['addr_clean'], ref),
    axis=1
)

df[['province_match_fuzzy', 'matched_province', 'province_score']] = df.apply(
    lambda x: best_province_match(x['addr_clean'], ref),
    axis=1
)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
any_mismatch = df[
    (df["city_match_fuzzy"] == False) |
    (df["province_match_fuzzy"] == False)
]

In [ ]:
df[df["city_match_fuzzy"] == False]["city_clean"].value_counts()

In [ ]:
df[df["city_match_fuzzy"] == False]["prov_clean"].value_counts()

In [ ]:
any_mismatch.to_excel("mismatches.xlsx", index=False)
